In [1]:
%pip install "transformers==4.56.2" "diffusers==0.32.2" "gradio==5.49.1" "huggingface-hub==0.36.2"

In [2]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM,TextStreamer, BitsAndBytesConfig
import torch
import gc




In [3]:
from getpass import getpass
from huggingface_hub import HfApi, hf_hub_download
from transformers import AutoTokenizer

hf_token = getpass("Hugging Face token: ")
LLAMA = "meta-llama/Llama-3.2-1B-Instruct"

# Confirm which account owns the token.
print("Authenticated as:", HfApi().whoami(token=hf_token)["name"])

# Check access with a small file before loading the model.
hf_hub_download(
    repo_id=LLAMA,
    filename="config.json",
    token=hf_token,
)
print("Model access confirmed")

tokenizer = AutoTokenizer.from_pretrained(
    LLAMA,
    token=hf_token,
)
tokenizer.pad_token = tokenizer.eos_token

Hugging Face token: ··········
Authenticated as: mestico01
Model access confirmed


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
# Instruct models and 1 reasoning model

# Llama 3.1 is larger and you should alreadybe approved
# see here: https://huggingface.co/meta-llama-3.1-8B-Instruct

# LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Llama 3.2 is smaller but you might need to request access again
# see here: https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

LLAMA = "meta-llama/Llama-3.2-1B-Instruct"
PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"



In [5]:
messages = [
    {
        "role": "system",
        "content": "Tell a joke for a room of Data Scientists"
    }
]

In [6]:
# Quantization Config - This allows us to load the model into memory and use less memory

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

PackageNotFoundError: No package metadata was found for bitsandbytes

In [ ]:
# Tokenizer

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages,
                                       return_tensors="pt").to("cuda")
